# Latent Space Evaluation — 4 Phases of XXZ+h

**Hamiltonian**: H = Σ_i (σ_x σ_x + σ_y σ_y + Δ σ_z σ_z) + h Σ_i σ_x

**Phase diagram** (axes: Δ vertical, h horizontal, h ∈ [0, 2.5]):
- **Neel_Z** (phase 1): Δ > 1, small h — Néel order along Z
- **FM_Z**   (phase 2): Δ < −1, small h — ferromagnetic order along Z
- **Neel_Y** (phase 3): −1 < Δ < 1, moderate h ∈ (0, ~1.5) — Néel order along Y (spin-flop)
- **PM_X**   (phase 4): h ≳ 2, right side — no LRO, magnetised along X

This notebook:
1. Loads `active_phase_discovery_checkpoint_latest.pkl`
2. Runs DMRG at representative (Δ, h) points in each phase
3. Encodes observables into the latent space
4. Visualises the latent space via PCA (analogous to `latent_eval.ipynb` in tfim_reorg)

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')

import jax
import jax.numpy as jnp
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from configs.config import DMRGConfig
from dmrg.hamiltonians import XXZhX
from dmrg.dmrg import run_dmrg
from models.autoencoder import fetch_latent

print('JAX devices:', jax.devices())

## 1. Load AE checkpoint

In [ ]:
CHECKPOINT = '../models/active_phase_discovery_checkpoint_latest.pkl'

with open(CHECKPOINT, 'rb') as f:
    ck = pickle.load(f)

ae_params = ck['params']
centroid  = np.array(ck['centroid'])   # reference latent vector (ferromagnetic region)
history   = ck.get('hist', {})

print('AE architecture:')
for i, (W, b) in enumerate(ae_params):
    print(f'  Layer {i}: W={np.array(W).shape}, b={np.array(b).shape}')
print(f'Centroid shape : {centroid.shape}')
print(f'Last (Δ, h)    : {ck["ham_param"]}')
print(f'Outer iter     : {ck.get("outer")}')
if history.get('ham_params'):
    print(f'History steps  : {len(history["ham_params"])}')

## 2. Parameter history

In [ ]:
if not history.get('ham_params'):
    print('No parameter history found in checkpoint.')
else:
    hist_params = np.array(history['ham_params'])   # (T, 2)
    hist_loss   = np.array(history['ham_loss'])      # (T,)
    steps       = np.arange(len(hist_params))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # ── subplot 1: Δ and h vs step ────────────────────────────────────────
    ax = axes[0]
    ax.plot(steps, hist_params[:, 0], label='Δ', color='steelblue')
    ax.plot(steps, hist_params[:, 1], label='h', color='darkorange')
    ax.set_xlabel('Optimizer step', fontsize=11)
    ax.set_ylabel('Parameter value', fontsize=11)
    ax.set_title('Δ and h over time', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # ── subplot 2: trajectory in (Δ, h) space ────────────────────────────
    ax2 = axes[1]
    T = len(hist_params)
    for t in range(T - 1):
        alpha = 0.3 + 0.6 * t / max(T - 1, 1)
        ax2.plot(hist_params[t:t+2, 1], hist_params[t:t+2, 0],
                 'k-', linewidth=1.0, alpha=alpha)
    ax2.scatter(hist_params[0,  1], hist_params[0,  0],
                c='lime',  s=120, marker='s', zorder=5, label='start')
    ax2.scatter(hist_params[-1, 1], hist_params[-1, 0],
                c='red',   s=120, marker='*', zorder=5, label='end')
    ax2.set_xlabel('h', fontsize=12)
    ax2.set_ylabel('Δ', fontsize=12)
    ax2.set_title('Trajectory in (Δ, h) space', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    # ── subplot 3: loss vs step ───────────────────────────────────────────
    ax3 = axes[2]
    ax3.plot(steps, hist_loss, color='firebrick', linewidth=1.0)
    ax3.set_xlabel('Optimizer step', fontsize=11)
    ax3.set_ylabel('Loss  ||z − z_centroid||²', fontsize=11)
    ax3.set_title('Latent distance loss over time', fontsize=12)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../figures/param_history.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Total steps : {len(hist_params)}')
    print(f'Start : Δ={hist_params[0,0]:.4f}, h={hist_params[0,1]:.4f}')
    print(f'End   : Δ={hist_params[-1,0]:.4f}, h={hist_params[-1,1]:.4f}')
    print(f'Δ range: [{hist_params[:,0].min():.4f}, {hist_params[:,0].max():.4f}]')
    print(f'h range: [{hist_params[:,1].min():.4f}, {hist_params[:,1].max():.4f}]')

## 3. Observables and DMRG config

In [ ]:
L = 20

sx = jnp.array([[0, 1],  [1, 0]],  dtype=jnp.float64)
sz = jnp.array([[1, 0],  [0, -1]], dtype=jnp.float64)
sy = jnp.array([[0, -1j],[1j, 0]], dtype=jnp.complex128)

# Must match the ordering used during training: Sz first, then Sx, then Sy
observables_list = (
    [(i, sz) for i in range(L)] +
    [(i, sx) for i in range(L)] +
    [(i, sy) for i in range(L)]
)
D = len(observables_list)
print(f'Observable dimension D = {D}')

dmrg_cfg = DMRGConfig()
print(f'DMRG: bond_dim={dmrg_cfg.max_bond}, sweeps={dmrg_cfg.sweeps}')

## 4. Define phase regions

Sample points well inside each phase to avoid boundary ambiguity.

In [ ]:
N_PER_PHASE = 20   # number of (Δ, h) samples per phase

phase_params = {
    # Phase 1 — Neel_Z: Δ > 1, small h (safely above the Δ=1 boundary)
    'Neel_Z': [(float(d), 0.0) for d in np.linspace(1.2, 2.0, N_PER_PHASE)],

    # Phase 2 — FM_Z: Δ < -1, small h (safely below the Δ=-1 boundary)
    'FM_Z':   [(float(d), 0.0) for d in np.linspace(-2.0, -1.2, N_PER_PHASE)],

    # Phase 3 — Neel_Y: sweep Δ ∈ [-0.7, 0.7] at fixed h=0.6 (middle of the spin-flop region)
    'Neel_Y': [(float(d), 0.6) for d in np.linspace(-0.7, 0.7, N_PER_PHASE)],

    # Phase 4 — PM_X: large h (h ∈ [2.0, 2.4]) at Δ=0, well inside the polarised phase
    'PM_X':   [(0.0, float(h)) for h in np.linspace(2.0, 2.4, N_PER_PHASE)],
}

for name, pts in phase_params.items():
    print(f'{name:7s}: {len(pts)} points  |  first={pts[0]}  last={pts[-1]}')

## 5. Run DMRG + encode latents

Results are cached to `latent_eval_data.pkl` so re-running is instant.

In [ ]:
CACHE_FILE = './latent_eval_data.pkl'

def encode_params_list(params_list, ae_params, observables_list, dmrg_cfg, L):
    """Run DMRG at each (Δ, h) and return stacked latent vectors."""
    latents = []
    for i, (delta, h) in enumerate(params_list):
        model = XXZhX(L, delta, h)
        psi   = run_dmrg(L, model, dmrg_cfg)

        obs_list = psi.get_site_exp_val(observables_list)
        obs = jnp.stack(
            [jnp.real(jnp.asarray(x)).reshape(()) for x in obs_list],
            axis=0
        )  # (D,)

        z = fetch_latent(ae_params, obs[None, :], jax.random.PRNGKey(0))[0]
        latents.append(np.array(z))

        if (i + 1) % 5 == 0 or i == 0:
            print(f'    {i+1:3d}/{len(params_list)}  (Δ={delta:.3f}, h={h:.3f})')
    return np.stack(latents)   # (N, latent_dim)


if os.path.exists(CACHE_FILE):
    print(f'Loading cached data from {CACHE_FILE} ...')
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    all_latents   = cached['all_latents']
    phase_params  = cached['phase_params']
    print('Done.')
else:
    print('Generating DMRG data (this takes a few minutes)...')
    all_latents = {}
    for phase_name, pts in phase_params.items():
        print(f'\n--- Phase: {phase_name} ({len(pts)} points) ---')
        all_latents[phase_name] = encode_params_list(
            pts, ae_params, observables_list, dmrg_cfg, L
        )

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump({'all_latents': all_latents, 'phase_params': phase_params}, f)
    print(f'\nSaved to {CACHE_FILE}')

for name, arr in all_latents.items():
    print(f'{name:4s}: shape={arr.shape},  mean_norm={np.linalg.norm(arr, axis=1).mean():.3f}')

In [ ]:
# Raw observable values for FM_Z — site-resolved <Sz>, <Sx>, <Sy>
# Sample 4 evenly-spaced points from the FM_Z sweep
fm_z_pts = phase_params['FM_Z']
indices   = [0, len(fm_z_pts)//3, 2*len(fm_z_pts)//3, len(fm_z_pts)-1]
selected  = [fm_z_pts[i] for i in indices]

sites = np.arange(L)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cmap = plt.cm.viridis
colors = [cmap(i / (len(selected) - 1)) for i in range(len(selected))]

obs_labels = ['⟨Sz⟩', '⟨Sx⟩', '⟨Sy⟩']

for k, (delta, h) in enumerate(selected):
    model = XXZhX(L, delta, h)
    psi   = run_dmrg(L, model, dmrg_cfg)

    obs_list = psi.get_site_exp_val(observables_list)
    obs = np.array([float(np.real(x)) for x in obs_list])  # (60,)

    # obs ordering: Sz (0:L), Sx (L:2L), Sy (2L:3L)
    sz_vals = obs[0:L]
    sx_vals = obs[L:2*L]
    sy_vals = obs[2*L:3*L]

    label = f'Δ={delta:.2f}, h={h:.1f}'
    axes[0].plot(sites, sz_vals, color=colors[k], marker='o', ms=3, label=label)
    axes[1].plot(sites, sx_vals, color=colors[k], marker='o', ms=3, label=label)
    axes[2].plot(sites, sy_vals, color=colors[k], marker='o', ms=3, label=label)

for ax, title in zip(axes, obs_labels):
    ax.set_xlabel('Site', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} across FM_Z sweep', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')

plt.suptitle('Raw observables for FM_Z phase (4 representative points)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../figures/fm_z_raw_observables.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. PCA of latent space — all 4 phases

In [ ]:
PHASE_ORDER   = ['Neel_Z', 'FM_Z', 'Neel_Y', 'PM_X']
PHASE_COLORS  = {'Neel_Z': '#d62728', 'FM_Z': '#1f77b4', 'Neel_Y': '#2ca02c', 'PM_X': '#ff7f0e'}
PHASE_MARKERS = {'Neel_Z': '^',       'FM_Z': 'o',       'Neel_Y': 's',       'PM_X': 'D'}

# Stack all latent vectors and fit PCA
combined = np.vstack([all_latents[p] for p in PHASE_ORDER])

pca = PCA(n_components=2)
reduced = pca.fit_transform(combined)

# Project the current centroid into PCA space
centroid_2d = pca.transform(centroid[None, :])[0]

fig, ax = plt.subplots(figsize=(6, 5))

idx = 0
for p in PHASE_ORDER:
    n   = len(all_latents[p])
    pts = reduced[idx:idx+n]
    ax.scatter(pts[:, 0], pts[:, 1],
               c=PHASE_COLORS[p], marker=PHASE_MARKERS[p],
               s=70, alpha=0.85, label=p, edgecolors='k', linewidths=0.4)
    idx += n

ax.scatter(*centroid_2d, c='black', marker='*', s=350, zorder=6,
           label='centroid', edgecolors='white', linewidths=0.5)

ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=11)
ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=11)
ax.set_title('PCA of Average Latent Space', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/latent_eval_pca_4phases.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PC1+PC2 total explained variance: {pca.explained_variance_ratio_.sum():.2%}')
print(f'Centroid PCA coords: PC1={centroid_2d[0]:.3f}, PC2={centroid_2d[1]:.3f}')

## 7. Optimizer trajectory in latent PCA space

Show where the active-phase-discovery optimizer has been, overlaid on the phase clusters.

In [ ]:
TRAJ_CACHE = './latent_eval_trajectory.pkl'

ham_params_history = history.get('ham_params', [])

if not ham_params_history:
    print('No trajectory history in checkpoint — skipping this cell.')
else:
    history_arr = np.array(ham_params_history)  # (T, 2)
    print(f'Trajectory: {len(history_arr)} steps')
    print(f'  Start : Δ={history_arr[0,0]:.3f}, h={history_arr[0,1]:.3f}')
    print(f'  End   : Δ={history_arr[-1,0]:.3f}, h={history_arr[-1,1]:.3f}')

    if os.path.exists(TRAJ_CACHE):
        print(f'Loading cached trajectory latents from {TRAJ_CACHE} ...')
        with open(TRAJ_CACHE, 'rb') as f:
            traj_cached = pickle.load(f)
        traj_latents = traj_cached['traj_latents']
        traj_subset  = traj_cached['traj_subset']
    else:
        # Subsample to ≤ 40 points to keep DMRG calls manageable
        stride      = max(1, len(history_arr) // 40)
        traj_subset = history_arr[::stride]   # (<=40, 2)
        print(f'Encoding {len(traj_subset)} trajectory points (stride={stride})...')
        traj_latents = encode_params_list(
            [(float(d), float(h)) for d, h in traj_subset],
            ae_params, observables_list, dmrg_cfg, L
        )
        with open(TRAJ_CACHE, 'wb') as f:
            pickle.dump({'traj_latents': traj_latents, 'traj_subset': traj_subset}, f)
        print(f'Saved to {TRAJ_CACHE}')

    traj_2d = pca.transform(traj_latents)  # project onto the PCA computed above

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left panel: trajectory in latent PCA space
    ax = axes[0]
    idx = 0
    for p in PHASE_ORDER:
        n   = len(all_latents[p])
        pts = reduced[idx:idx+n]
        ax.scatter(pts[:, 0], pts[:, 1],
                   c=PHASE_COLORS[p], marker=PHASE_MARKERS[p],
                   s=55, alpha=0.5, label=p, edgecolors='k', linewidths=0.3)
        idx += n

    # Trajectory line with colour gradient (early=light, late=dark)
    T = len(traj_2d)
    for t in range(T - 1):
        alpha = 0.4 + 0.5 * t / max(T - 1, 1)
        ax.plot(traj_2d[t:t+2, 0], traj_2d[t:t+2, 1],
                'k-', linewidth=1.5, alpha=alpha)

    ax.scatter(traj_2d[0, 0],  traj_2d[0, 1],
               c='lime', s=150, marker='s', zorder=6, label='traj start')
    ax.scatter(traj_2d[-1, 0], traj_2d[-1, 1],
               c='black', s=200, marker='*', zorder=6, label='traj end')
    ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
    ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
    ax.set_title('Trajectory in Latent PCA Space', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Right panel: trajectory in (Δ, h) parameter space
    ax2 = axes[1]
    for p in PHASE_ORDER:
        pts = np.array(phase_params[p])
        ax2.scatter(pts[:, 0], pts[:, 1],
                    c=PHASE_COLORS[p], marker=PHASE_MARKERS[p],
                    s=55, alpha=0.5, label=p, edgecolors='k', linewidths=0.3)

    T = len(traj_subset)
    for t in range(T - 1):
        alpha = 0.4 + 0.5 * t / max(T - 1, 1)
        ax2.plot(traj_subset[t:t+2, 0], traj_subset[t:t+2, 1],
                 'k-', linewidth=1.5, alpha=alpha)

    ax2.scatter(traj_subset[0, 0],  traj_subset[0, 1],
                c='lime',  s=150, marker='s', zorder=6, label='start')
    ax2.scatter(traj_subset[-1, 0], traj_subset[-1, 1],
                c='black', s=200, marker='*', zorder=6, label='end')
    ax2.set_xlabel('Δ', fontsize=12)
    ax2.set_ylabel('h', fontsize=12)
    ax2.set_title('Trajectory in (Δ, h) Parameter Space', fontsize=12)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../figures/latent_eval_trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Pairwise latent distances between phases

How well-separated are the phases in latent space?

In [ ]:
# Mean latent vector per phase
means = {p: all_latents[p].mean(axis=0) for p in PHASE_ORDER}

print('Pairwise L2 distances between phase centroids (in full latent space):')
print(f'{"":8s}', '  '.join(f'{p:7s}' for p in PHASE_ORDER))
for p1 in PHASE_ORDER:
    row = [f'{np.linalg.norm(means[p1] - means[p2]):7.3f}' for p2 in PHASE_ORDER]
    print(f'{p1:8s}', '  '.join(row))

print()
print('Distance from each phase centroid to the AE centroid (FM_Z reference):')
for p in PHASE_ORDER:
    d = np.linalg.norm(means[p] - centroid)
    print(f'  {p:8s}  →  {d:.4f}')